# Week 11: Project Structure & Imports — PHASE 6: Scaling Your Pipeline
*Core Mastery: "I can organize code across files and import helper functions in Colab"*

*Computer Programming II | 5 Hours | Dr. Arif Solmaz*

## 🎯 Learning Objectives

By the end of this week, you will be able to:

1. Explain what a Python module is and how it differs from a script
2. Create `.py` files directly from Google Colab using `%%writefile`
3. Use `import`, `from ... import`, and `import ... as` correctly
4. Understand how `sys.path` works and modify it when needed
5. Design a clean project directory structure with `src/`, `data/`, `outputs/`
6. Build a reusable helper module library
7. Use the `__name__ == '__main__'` guard correctly
8. Download files from URLs programmatically
9. Create a multi-module pipeline that imports from custom modules
10. Prepare a project skeleton for the mini project

## 🎯 Core Mastery Connection

PHASE 6: Scaling Your Pipeline — This week's topic directly supports the course's core mastery goal: *"I can organize code across files and import helper functions in Colab"*.

As your projects grow beyond a single notebook, you need to **organize code into reusable modules**. This week teaches the skills that separate beginner scripts from professional-quality projects.

| Skill | Why It Matters |
|---|---|
| Creating modules | Reuse code without copy-paste |
| Import system | Load the right code from the right place |
| Directory structure | Keep projects organized and maintainable |
| `__name__` guard | Modules that work both standalone and imported |
| Multi-module pipelines | Scale to real-world project complexity |

---
## 🤝 Mechatronics Learning Contract

- **Professional relevance:** examples and core exercises model the data, sensing, automation, numerical, and decision tasks used in mechatronics engineering.
- **Interaction:** predict before running, compare reasoning with a partner, and ask whenever a step is unclear; scheduled checkpoints guarantee question time.
- **Assessment alignment:** worked examples and Core Exercises 1–8 rehearse the same reasoning operations used on exams—trace, implement, debug, interpret, and justify—while exam values and contexts may change.
- **Learning evidence:** weekly notebooks remain private practice. Non-exam evidence comes from scheduled in-class project demonstrations/presentations using a published rubric, not homework collection.


---
## 🧭 Five-Hour Class Roadmap

This notebook is designed for one five-hour class with four short breaks.

| Target | Activity |
|---|---|
| 00:00–00:55 | Concepts and examples → Checkpoint 1 |
| 00:55–01:05 | Break |
| 01:05–01:55 | Concepts and examples → Checkpoint 2 |
| 01:55–02:05 | Break |
| 02:05–02:55 | Concepts and examples → Checkpoint 3 |
| 02:55–03:05 | Break |
| 03:05–03:55 | Concepts and examples → Checkpoint 4 |
| 03:55–04:05 | Break |
| 04:05–04:45 | Core Practice (Exercises 1–8) → Checkpoint 5 |
| 04:45–05:00 | Review and retry failed checks |

Checkpoints provide immediate feedback only inside your Colab runtime. Nothing
is transmitted, saved for grading, or reviewed by the instructor. Exercises 9
and above are optional extensions—not homework.


In [ ]:
# Run this setup cell once at the start of class.
_checkpoint_results = {}

def check_answer(number, answer, expected, explanation):
    actual = str(answer).strip().lower().replace(" ", "")
    target = str(expected).strip().lower().replace(" ", "")
    correct = actual == target
    _checkpoint_results[int(number)] = (int(correct), 1)
    if correct:
        print(f"✅ Checkpoint {number}: correct")
        print("Why:", explanation)
    elif not str(answer).strip():
        print(f"🟡 Checkpoint {number}: enter an answer, then run this cell again.")
    else:
        print(f"🔴 Checkpoint {number}: not yet. Review the preceding examples and retry.")
    return correct

def exercise_checkpoint(number, expected=8):
    """Count core exercise cells that contain work and were run in this runtime."""
    import re
    completed = set()
    for source in globals().get("In", []):
        match = re.search(r"#\s*✏️\s*\[EX(\d+)\]", str(source), flags=re.I)
        if not match:
            continue
        answer = re.sub(r"^.*?#\s*✏️\s*\[EX\d+\]", "", str(source), count=1, flags=re.I | re.S).strip()
        if answer and answer != "pass" and "your code here" not in answer.lower():
            completed.add(int(match.group(1)))
    checks = [(index, index in completed) for index in range(1, expected + 1)]
    passed = sum(done for _, done in checks)
    _checkpoint_results[int(number)] = (passed, expected)
    print(f"Core Practice: {passed}/{expected} exercise cells edited and run")
    missing = [str(index) for index, done in checks if not done]
    if not missing:
        print("✅ Core Practice complete.")
    else:
        print("🟡 Still to complete/run:", ", ".join(missing))
    return passed, expected

def show_progress_summary():
    print("\n=== My local progress ===")
    for number in range(1, 6):
        if number in _checkpoint_results:
            passed, total = _checkpoint_results[number]
            print(f"Checkpoint {number}: {passed}/{total}")
        else:
            print(f"Checkpoint {number}: not run")
    print("Results exist only in this temporary runtime.")

print("✅ Local self-check tools ready")


---
## Part 1: What is a Module?

### Definition

A **module** is simply a `.py` file containing Python code (functions, classes, variables). When you `import` a module, Python executes the file and makes its contents available.

### Module vs Script

| | Module | Script |
|---|---|---|
| **Purpose** | Provide reusable functions/classes | Run a specific task |
| **How it runs** | Imported by other code | Executed directly |
| **File extension** | `.py` | `.py` |
| **Contains** | Functions, classes, constants | Top-level logic, `print()` calls |
| **Example** | `math`, `os`, `json` | `main.py`, `run_analysis.py` |

### Built-in Modules You Already Know

```python
import math          # math.sqrt(), math.pi
import random        # random.randint(), random.choice()
import os            # os.path.exists(), os.listdir()
import json          # json.loads(), json.dumps()
import time          # time.time(), time.perf_counter()
```

Each of these is a `.py` file (or C extension) that lives somewhere on your system. Today you will learn to create your own!

**Figure 11.1** — Exploring a built-in module

In [ ]:
import math

# See what's inside a module
print("Module name:", math.__name__)
print("Module file:", math.__file__)
print()

# List all public names
public_names = [name for name in dir(math) if not name.startswith('_')]
print(f"math has {len(public_names)} public names:")
print(", ".join(public_names[:15]), "...")

**Figure 11.2** — Checking where a module lives

In [ ]:
import os
import json
import random

for mod in [os, json, random]:
    print(f"{mod.__name__:10s} → {mod.__file__}")

---
## Part 2: Creating `.py` Files in Colab

### The `%%writefile` Magic Command

In Google Colab (and Jupyter), the `%%writefile` magic command writes the cell's content to a file on disk. This is how you create modules without leaving the notebook.

### Syntax

```python
%%writefile filename.py
# Python code goes here
def my_function():
    return "Hello from my module!"
```

### Important Notes

| Point | Detail |
|---|---|
| `%%writefile` must be first line | No code or comments before it |
| Creates / overwrites the file | Careful with existing files |
| File is saved in current directory | Use `!pwd` to check |
| File is NOT automatically imported | You must `import` it separately |
| `-a` flag appends | `%%writefile -a file.py` adds to existing file |

**Figure 11.3** — Creating your first module with `%%writefile`

In [ ]:
%%writefile helpers.py
"""helpers.py — Utility functions for data processing.

Author: Computer Programming II Course
"""

def clean_value(x, lo=0, hi=100):
    """Clamp x to [lo, hi] range."""
    return max(lo, min(hi, x))

def mean(values):
    """Return the arithmetic mean."""
    if not values:
        return 0.0
    return sum(values) / len(values)

def filter_valid(data, lo=0, hi=100):
    """Keep only values in (lo, hi) range."""
    return [x for x in data if lo < x < hi]

# Module-level constant
VERSION = "1.0.0"

print(f"helpers.py loaded (v{VERSION})")

**Figure 11.4** — Importing and using the module

In [ ]:
import helpers

# Use functions from the module
data = [23.5, -5, 102, 45.2, 67.8, 200, 33.1]

cleaned = [helpers.clean_value(x) for x in data]
valid = helpers.filter_valid(data, lo=0, hi=100)
avg = helpers.mean(valid)

print(f"Original : {data}")
print(f"Cleaned  : {cleaned}")
print(f"Valid    : {valid}")
print(f"Mean     : {avg:.2f}")
print(f"Version  : {helpers.VERSION}")

**Figure 11.5** — Using `%%writefile -a` to append to a file

In [ ]:
%%writefile -a helpers.py

def median(values):
    """Return the median value."""
    if not values:
        return 0.0
    s = sorted(values)
    n = len(s)
    if n % 2 == 1:
        return s[n // 2]
    return (s[n // 2 - 1] + s[n // 2]) / 2

> **Important:** After modifying a module, you need to reload it. Python caches imported modules.

In [ ]:
# Reload the module to pick up changes
import importlib
importlib.reload(helpers)

# Now median is available
print(helpers.median([3, 1, 4, 1, 5, 9, 2, 6]))

---
### ⏱️ Checkpoint 1 of 5 — Modules (target 00:55)

What file extension does a Python module normally use?

Enter a short answer in the next cell and run it. Retry after reviewing the
preceding examples if needed.


In [ ]:
checkpoint_1_answer = ""  # enter your answer
check_answer(
    1, checkpoint_1_answer, '.py',
    'A module is commonly a `.py` file.',
)


---
## Part 3: The `import` Statement — Variations

### Five Ways to Import

| Syntax | What You Get | Usage |
|---|---|---|
| `import helpers` | Full module | `helpers.mean()` |
| `from helpers import mean` | Just `mean` | `mean()` |
| `from helpers import mean, median` | Multiple names | `mean()`, `median()` |
| `from helpers import *` | Everything (avoid!) | `mean()` |
| `import helpers as h` | Alias | `h.mean()` |

### Best Practices

| Practice | Reason |
|---|---|
| **Prefer `import module`** | Clear where each name comes from |
| **Use `from` for frequently used names** | Less typing for common functions |
| **Avoid `from module import *`** | Pollutes namespace, hides origin |
| **Use aliases for long names** | `import numpy as np` is conventional |
| **Group imports at top of file** | Easy to see all dependencies |

### Import Order Convention (PEP 8)

```python
# 1. Standard library
import os
import json
import time

# 2. Third-party packages
import numpy as np
import matplotlib.pyplot as plt

# 3. Local / project modules
import helpers
from pipeline import clean_stage
```

**Figure 11.6** — All five import styles in action

In [ ]:
# Style 1: import module
import helpers
print("Style 1:", helpers.mean([1, 2, 3]))

# Style 2: from module import name
from helpers import mean
print("Style 2:", mean([4, 5, 6]))

# Style 3: from module import multiple
from helpers import mean, median, filter_valid
print("Style 3:", mean([7, 8, 9]), median([7, 8, 9]))

# Style 4: import with alias
import helpers as h
print("Style 4:", h.mean([10, 11, 12]))

# Style 5: from module import * (AVOID in production)
# from helpers import *
# print("Style 5:", mean([13, 14, 15]))
print("Style 5: Skipped — avoid 'import *' in production code!")

**Figure 11.7** — What happens when you import

In [ ]:
import sys

# Check if helpers is already cached
print("helpers in sys.modules?", "helpers" in sys.modules)

# First import: Python executes the file
import helpers  # will print "helpers.py loaded" only on first import

# Second import: Python uses the cached version
import helpers  # no output — already cached!

# Force reload
import importlib
importlib.reload(helpers)  # re-executes the file

print("\nPython caches modules in sys.modules to avoid re-executing them.")

---
## Part 4: `sys.path` and How Python Finds Modules

### The Module Search Order

When you write `import helpers`, Python searches these locations **in order**:

1. **`sys.modules` cache** — already imported? Use the cached version
2. **Built-in modules** — `sys`, `os`, `math`, etc.
3. **`sys.path` directories** — a list of directories to search

### What's in `sys.path`?

`sys.path` is a Python list of directory paths. You can inspect and modify it.

| Default Entry | Meaning |
|---|---|
| `""` (empty string) | Current working directory |
| Python standard library paths | Where built-in modules live |
| `site-packages` | Where pip-installed packages live |

### Common Problem

If your module is in a subdirectory (e.g., `src/helpers.py`), Python won't find it unless `src/` is in `sys.path`.

**Figure 11.8** — Inspecting `sys.path`

In [ ]:
import sys

print("Python searches these directories for modules:\n")
for i, path in enumerate(sys.path):
    label = "(current dir)" if path == "" else ""
    print(f"  [{i}] {path or '""'} {label}")

**Figure 11.9** — Adding a custom directory to `sys.path`

In [ ]:
import sys
import os

# Create a subdirectory
os.makedirs("src", exist_ok=True)

# Check if 'src' is already in sys.path
if "src" not in sys.path and os.path.abspath("src") not in sys.path:
    sys.path.insert(0, os.path.abspath("src"))
    print(f"Added {os.path.abspath('src')} to sys.path")
else:
    print("src already in sys.path")

# Verify
print(f"\nFirst 3 entries in sys.path:")
for p in sys.path[:3]:
    print(f"  {p}")

**Figure 11.10** — Creating a module in a subdirectory and importing it

In [ ]:
%%writefile src/stats_utils.py
"""stats_utils.py — Statistical utility functions."""

def variance(values):
    """Compute population variance."""
    if len(values) < 2:
        return 0.0
    m = sum(values) / len(values)
    return sum((x - m)**2 for x in values) / len(values)

def std_dev(values):
    """Compute population standard deviation."""
    return variance(values) ** 0.5

def z_scores(values):
    """Return z-scores for each value."""
    m = sum(values) / len(values)
    s = std_dev(values)
    if s == 0:
        return [0.0] * len(values)
    return [(x - m) / s for x in values]

print("stats_utils loaded from src/")

In [ ]:
# Now import from the src directory
import stats_utils

data = [22.1, 24.5, 19.8, 23.7, 25.0, 21.3, 22.9]
print(f"Data: {data}")
print(f"Variance  : {stats_utils.variance(data):.4f}")
print(f"Std Dev   : {stats_utils.std_dev(data):.4f}")
print(f"Z-scores  : {[f'{z:.2f}' for z in stats_utils.z_scores(data)]}")

---
### ⏱️ Checkpoint 2 of 5 — Imports (target 01:55)

Does importing a module execute its top-level statements? Answer yes or no.

Enter a short answer in the next cell and run it. Retry after reviewing the
preceding examples if needed.


In [ ]:
checkpoint_2_answer = ""  # enter your answer
check_answer(
    2, checkpoint_2_answer, 'yes',
    'Top-level module code runs on first import.',
)


---
## Part 5: Project Directory Structure

### Why Structure Matters

| Without Structure | With Structure |
|---|---|
| All files in one folder | Logical grouping |
| Hard to find things | Easy navigation |
| Name collisions | Clear namespaces |
| Can't reuse code | Modules are reusable |
| Messy for collaborators | Professional and clear |

### Recommended Layout

```
my_project/
├── src/               # Source code modules
│   ├── loader.py      # Data loading functions
│   ├── cleaner.py     # Data cleaning functions
│   ├── analyzer.py    # Analysis / computation functions
│   └── reporter.py    # Output formatting functions
├── data/              # Input data files
│   ├── raw/           # Original, untouched data
│   └── processed/     # Cleaned data
├── outputs/           # Generated outputs
│   ├── figures/       # Plots and charts
│   └── reports/       # Text reports
├── tests/             # Test files
│   └── test_loader.py
├── main.py            # Main entry point
└── README.md          # Project description
```

### In Colab

Since Colab starts fresh each session, you create this structure at the beginning of your notebook using `os.makedirs()` and `%%writefile`.

**Figure 11.11** — Creating a project structure in Colab

In [ ]:
import os

# Define project root
PROJECT = "earthquake_analysis"

# Create directory structure
dirs = [
    f"{PROJECT}/src",
    f"{PROJECT}/data/raw",
    f"{PROJECT}/data/processed",
    f"{PROJECT}/outputs/figures",
    f"{PROJECT}/outputs/reports",
    f"{PROJECT}/tests",
]

for d in dirs:
    os.makedirs(d, exist_ok=True)
    print(f"  Created: {d}/")

print(f"\n✅ Project structure ready at ./{PROJECT}/")

**Figure 11.12** — Visualizing the project tree

In [ ]:
import os

def print_tree(directory, prefix=""):
    """Print a directory tree."""
    entries = sorted(os.listdir(directory))
    dirs = [e for e in entries if os.path.isdir(os.path.join(directory, e))]
    files = [e for e in entries if os.path.isfile(os.path.join(directory, e))]

    for f in files:
        print(f"{prefix}├── {f}")
    for i, d in enumerate(dirs):
        is_last = (i == len(dirs) - 1) and not files
        connector = "└── " if is_last and i == len(dirs)-1 else "├── "
        print(f"{prefix}{connector}{d}/")
        extension = "    " if connector == "└── " else "│   "
        print_tree(os.path.join(directory, d), prefix + extension)

print(f"{PROJECT}/")
print_tree(PROJECT)

**Figure 11.13** — Adding `sys.path` for project imports

In [ ]:
import sys
import os

# Add the project's src directory to the import path
src_path = os.path.abspath(f"{PROJECT}/src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)
    print(f"Added to sys.path: {src_path}")

# Also add the project root
root_path = os.path.abspath(PROJECT)
if root_path not in sys.path:
    sys.path.insert(0, root_path)
    print(f"Added to sys.path: {root_path}")

print("\n✅ Import paths configured.")

---
## Part 6: Creating a Helper Module Library

### Strategy

Build a library of small, focused modules — each responsible for one task:

| Module | Responsibility | Key Functions |
|---|---|---|
| `loader.py` | Read data from files/URLs | `load_csv()`, `download()` |
| `cleaner.py` | Validate and clean data | `filter_valid()`, `remove_outliers()` |
| `analyzer.py` | Compute statistics | `mean()`, `std_dev()`, `correlate()` |
| `reporter.py` | Format output | `make_table()`, `save_report()` |

### Benefits

- **Reuse**: Import the same module in many notebooks
- **Testing**: Test each module independently
- **Readability**: Small files are easier to understand
- **Collaboration**: Different people can work on different modules

**Figure 11.14** — Creating `loader.py`

In [ ]:
%%writefile earthquake_analysis/src/loader.py
"""loader.py — Data loading utilities."""

import csv
import os
import random

def load_csv(filepath, delimiter=','):
    """Load CSV file and return list of dicts."""
    if not os.path.exists(filepath):
        raise FileNotFoundError(f"File not found: {filepath}")
    with open(filepath, 'r', encoding='utf-8') as f:
        reader = csv.DictReader(f, delimiter=delimiter)
        return list(reader)

def generate_sample_data(n=1000, seed=42):
    """Generate sample earthquake data for testing."""
    random.seed(seed)
    data = []
    cities = ["Istanbul", "Ankara", "Izmir", "Bursa", "Antalya",
              "Adana", "Konya", "Kayseri", "Trabzon", "Van"]
    for i in range(n):
        data.append({
            "id": i + 1,
            "city": random.choice(cities),
            "magnitude": round(random.uniform(0.5, 8.0), 1),
            "depth_km": round(random.uniform(1, 150), 1),
            "year": random.randint(2000, 2024),
        })
    return data

def save_csv(data, filepath, fieldnames=None):
    """Save list of dicts to CSV."""
    if not data:
        return
    if fieldnames is None:
        fieldnames = list(data[0].keys())
    os.makedirs(os.path.dirname(filepath), exist_ok=True)
    with open(filepath, 'w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(data)

print("loader.py loaded")

**Figure 11.15** — Creating `cleaner.py`

In [ ]:
%%writefile earthquake_analysis/src/cleaner.py
"""cleaner.py — Data cleaning utilities."""

def filter_by_range(data, field, lo, hi):
    """Keep records where lo <= field value <= hi."""
    result = []
    for row in data:
        val = float(row[field])
        if lo <= val <= hi:
            result.append(row)
    return result

def remove_duplicates(data, key_field):
    """Remove duplicate records based on a key field."""
    seen = set()
    result = []
    for row in data:
        key = row[key_field]
        if key not in seen:
            seen.add(key)
            result.append(row)
    return result

def fill_missing(data, field, default):
    """Replace missing/empty values with a default."""
    for row in data:
        if field not in row or row[field] == '' or row[field] is None:
            row[field] = default
    return data

print("cleaner.py loaded")

**Figure 11.16** — Creating `analyzer.py`

In [ ]:
%%writefile earthquake_analysis/src/analyzer.py
"""analyzer.py — Statistical analysis functions."""

from collections import Counter

def extract_field(data, field):
    """Extract a list of float values from a field."""
    return [float(row[field]) for row in data]

def basic_stats(values):
    """Compute basic statistics."""
    if not values:
        return {"count": 0, "mean": 0, "min": 0, "max": 0, "std": 0}
    n = len(values)
    m = sum(values) / n
    var = sum((x - m)**2 for x in values) / n
    return {
        "count": n,
        "mean": round(m, 4),
        "min": round(min(values), 4),
        "max": round(max(values), 4),
        "std": round(var ** 0.5, 4),
    }

def group_by(data, field):
    """Group records by a field value. Returns dict of lists."""
    groups = {}
    for row in data:
        key = row[field]
        if key not in groups:
            groups[key] = []
        groups[key].append(row)
    return groups

def frequency_table(data, field):
    """Count occurrences of each value in a field."""
    values = [row[field] for row in data]
    return dict(Counter(values).most_common())

print("analyzer.py loaded")

**Figure 11.17** — Creating `reporter.py`

In [ ]:
%%writefile earthquake_analysis/src/reporter.py
"""reporter.py — Output formatting utilities."""

import os

def make_table(headers, rows, col_width=15):
    """Create a formatted text table."""
    header_line = " | ".join(f"{h:>{col_width}}" for h in headers)
    separator = "-+-".join("-" * col_width for _ in headers)
    lines = [header_line, separator]
    for row in rows:
        lines.append(" | ".join(f"{str(v):>{col_width}}" for v in row))
    return "\n".join(lines)

def save_report(text, filepath):
    """Save a text report to a file."""
    os.makedirs(os.path.dirname(filepath), exist_ok=True)
    with open(filepath, 'w', encoding='utf-8') as f:
        f.write(text)
    print(f"Report saved to {filepath}")

def summary_block(title, stats_dict):
    """Create a formatted summary block."""
    lines = [f"=== {title} ==="]
    for key, val in stats_dict.items():
        lines.append(f"  {key:15s}: {val}")
    lines.append("=" * (len(title) + 8))
    return "\n".join(lines)

print("reporter.py loaded")

**Figure 11.18** — Using all four modules together

In [ ]:
import importlib
import sys

# Ensure src is in path
src_path = os.path.abspath("earthquake_analysis/src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

# Import our modules
import loader
import cleaner
import analyzer
import reporter

# Reload in case they were previously imported
for mod in [loader, cleaner, analyzer, reporter]:
    importlib.reload(mod)

# === PIPELINE ===

# Stage 1: Load
data = loader.generate_sample_data(n=5000)
print(f"Loaded {len(data)} records")

# Stage 2: Clean
data = cleaner.filter_by_range(data, "magnitude", 2.0, 7.0)
print(f"After filtering: {len(data)} records")

# Stage 3: Analyze
magnitudes = analyzer.extract_field(data, "magnitude")
stats = analyzer.basic_stats(magnitudes)
print(f"Stats: {stats}")

city_freq = analyzer.frequency_table(data, "city")
print(f"City distribution: {city_freq}")

# Stage 4: Report
table = reporter.make_table(
    ["City", "Count"],
    [[city, count] for city, count in sorted(city_freq.items())]
)
print(f"\n{table}")

---
### ⏱️ Checkpoint 3 of 5 — Main guard (target 02:55)

What variable is checked by the main guard?

Enter a short answer in the next cell and run it. Retry after reviewing the
preceding examples if needed.


In [ ]:
checkpoint_3_answer = ""  # enter your answer
check_answer(
    3, checkpoint_3_answer, '__name__',
    "The guard compares `__name__` with `'__main__'`.",
)


---
## Part 7: The `__name__ == '__main__'` Guard

### The Problem

When Python imports a module, it **executes all the code** in that file. If the module contains test code or `print()` statements at the top level, they run on import — which is usually unwanted.

### The Solution

```python
if __name__ == '__main__':
    # This code runs ONLY when the file is executed directly
    # It does NOT run when the file is imported
    print("Running tests...")
```

### How It Works

| Scenario | `__name__` value |
|---|---|
| File executed directly (`python file.py`) | `'__main__'` |
| File imported (`import file`) | `'file'` (the module name) |

### Why It Matters

| Without Guard | With Guard |
|---|---|
| Tests run on import | Tests only run when you want |
| Print statements on import | Clean, silent import |
| Side effects on import | Predictable behaviour |

**Figure 11.19** — Module with and without `__name__` guard

In [ ]:
%%writefile earthquake_analysis/src/math_tools.py
"""math_tools.py — Mathematical utility functions."""

def factorial(n):
    """Compute n! iteratively."""
    result = 1
    for i in range(2, n + 1):
        result *= i
    return result

def is_prime(n):
    """Check if n is prime."""
    if n < 2:
        return False
    for i in range(2, int(n**0.5) + 1):
        if n % i == 0:
            return False
    return True

def fibonacci(n):
    """Return first n Fibonacci numbers."""
    if n <= 0:
        return []
    if n == 1:
        return [0]
    fibs = [0, 1]
    for _ in range(2, n):
        fibs.append(fibs[-1] + fibs[-2])
    return fibs

# ── This block runs ONLY when executed directly ──
if __name__ == '__main__':
    print("Running math_tools.py self-tests...")
    assert factorial(5) == 120
    assert factorial(0) == 1
    assert is_prime(17) == True
    assert is_prime(4) == False
    assert fibonacci(7) == [0, 1, 1, 2, 3, 5, 8]
    print("✅ All self-tests passed!")

In [ ]:
# When we IMPORT, the if __name__ == '__main__' block does NOT run
import math_tools
importlib.reload(math_tools)

# But we can use the functions
print(f"10! = {math_tools.factorial(10)}")
print(f"Is 97 prime? {math_tools.is_prime(97)}")
print(f"First 10 Fibonacci: {math_tools.fibonacci(10)}")
print("\nNotice: 'Running self-tests...' did NOT print — the guard worked!")

**Figure 11.20** — Running a module directly to test it

In [ ]:
# In Colab, we can run a .py file directly using !python
!python earthquake_analysis/src/math_tools.py

> **Tip:** Always add the `__name__` guard to your modules. It's a professional Python practice and lets you include self-tests without them running on import.

---
## Part 8: Preparing for the Mini Project

### Mini Project Preview

In the coming weeks, you will build a complete data analysis pipeline as a mini project. This week, you prepare the **skeleton** — the directory structure, modules, and import setup.

### Project Skeleton Checklist

| Item | Status |
|---|---|
| Directory structure created | `src/`, `data/`, `outputs/` |
| Helper modules written | `loader.py`, `cleaner.py`, `analyzer.py`, `reporter.py` |
| `sys.path` configured | Can import from `src/` |
| `__name__` guards added | All modules have guards |
| Sample data generated | Testing data available |
| Basic pipeline tested | All stages work end-to-end |

### Downloading Data in Colab

For real projects, you often need to download data from the internet. Here's how:

**Figure 11.21** — Downloading files from a URL

In [ ]:
import urllib.request
import os

def download_file(url, save_path):
    """Download a file from a URL to a local path."""
    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    print(f"Downloading {url}...")
    try:
        urllib.request.urlretrieve(url, save_path)
        size = os.path.getsize(save_path)
        print(f"✅ Saved to {save_path} ({size:,} bytes)")
        return True
    except Exception as e:
        print(f"❌ Download failed: {e}")
        return False

# Example: download a small test file
# download_file(
#     "https://raw.githubusercontent.com/datasets/country-codes/master/data/country-codes.csv",
#     "earthquake_analysis/data/raw/countries.csv"
# )

# For this demo, let's create a sample file instead
data = loader.generate_sample_data(n=2000, seed=123)
save_path = "earthquake_analysis/data/raw/earthquakes.csv"
loader.save_csv(data, save_path)
print(f"Generated sample data at {save_path}")

**Figure 11.22** — Complete project pipeline using imports

In [ ]:
import sys, os, importlib

# Setup paths
src_path = os.path.abspath("earthquake_analysis/src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

import loader, cleaner, analyzer, reporter
for m in [loader, cleaner, analyzer, reporter]:
    importlib.reload(m)

print("=" * 60)
print("  EARTHQUAKE ANALYSIS PIPELINE")
print("=" * 60)

# Step 1: Load
data = loader.generate_sample_data(n=10000, seed=42)
print(f"\n[1] Loaded {len(data)} records")

# Step 2: Clean
data = cleaner.filter_by_range(data, "magnitude", 2.0, 7.5)
data = cleaner.filter_by_range(data, "depth_km", 1, 100)
print(f"[2] After cleaning: {len(data)} records")

# Step 3: Analyze
magnitudes = analyzer.extract_field(data, "magnitude")
mag_stats = analyzer.basic_stats(magnitudes)
city_groups = analyzer.group_by(data, "city")
city_freq = analyzer.frequency_table(data, "city")

print(f"[3] Magnitude stats: mean={mag_stats['mean']}, std={mag_stats['std']}")

# Step 4: Report
report_lines = []
report_lines.append("EARTHQUAKE ANALYSIS REPORT")
report_lines.append("=" * 40)
report_lines.append(reporter.summary_block("Magnitude Statistics", mag_stats))
report_lines.append("")

city_rows = sorted(city_freq.items(), key=lambda x: -x[1])
report_lines.append(reporter.make_table(
    ["City", "Count", "Percentage"],
    [[city, count, f"{count/len(data)*100:.1f}%"] for city, count in city_rows]
))

report_text = "\n".join(report_lines)
reporter.save_report(report_text, "earthquake_analysis/outputs/reports/summary.txt")

print(f"[4] Report saved!")
print(f"\n{report_text}")

**Figure 11.23** — Verifying the project structure after pipeline run

In [ ]:
print(f"\nFinal project structure:")
print(f"{PROJECT}/")
print_tree(PROJECT)

**Figure 11.24** — A main.py entry point

In [ ]:
%%writefile earthquake_analysis/main.py
"""main.py — Entry point for the earthquake analysis pipeline."""

import sys
import os

# Add src to path
sys.path.insert(0, os.path.join(os.path.dirname(__file__), "src"))

import loader
import cleaner
import analyzer
import reporter

def run_pipeline(n=5000):
    """Run the complete analysis pipeline."""
    print("=== Earthquake Analysis Pipeline ===\n")

    # Load
    data = loader.generate_sample_data(n=n, seed=42)
    print(f"[Load]    {len(data)} records generated")

    # Clean
    data = cleaner.filter_by_range(data, "magnitude", 2.0, 7.5)
    print(f"[Clean]   {len(data)} records after filtering")

    # Analyze
    magnitudes = analyzer.extract_field(data, "magnitude")
    stats = analyzer.basic_stats(magnitudes)
    print(f"[Analyze] Mean magnitude: {stats['mean']}")

    # Report
    summary = reporter.summary_block("Results", stats)
    report_path = os.path.join(os.path.dirname(__file__), "outputs", "reports", "auto_report.txt")
    reporter.save_report(summary, report_path)
    print(f"[Report]  Saved to {report_path}")

    return stats

if __name__ == '__main__':
    run_pipeline()

In [ ]:
# Run main.py directly
!python earthquake_analysis/main.py

---
### ⏱️ Checkpoint 4 of 5 — Paths (target 03:55)

Should project code rely on one person's absolute path? Answer yes or no.

Enter a short answer in the next cell and run it. Retry after reviewing the
preceding examples if needed.


In [ ]:
checkpoint_4_answer = ""  # enter your answer
check_answer(
    4, checkpoint_4_answer, 'no',
    'Portable projects use relative/configured paths.',
)


---
## Exercises

Complete the exercises below. Write your code in the provided code cells.

> **Each exercise is a problem. Think about the STEPS before you code.** What data do you need? What calculations? What output? Plan your steps first, then translate them into Python.

> Run each cell after writing your solution to check if it works!

### Core Practice and Optional Extension

- **Exercises 1–8:** core in-class practice.
- **Exercises 9 and above:** optional extension; these are not homework.
- Run each completed code cell so Checkpoint 5 can count your local progress.


### Exercise 1: Create a Helper Module (Easy)

Create a file called `string_utils.py` using `%%writefile` that contains:
1. `clean_text(s)` — strip whitespace and convert to lowercase
2. `count_words(s)` — return the number of words
3. `truncate(s, max_len=50)` — truncate string to max_len chars, add "..." if truncated

Import and test all three functions.

<details>
<summary>💡 Hint</summary>
Use `s.strip().lower()`, `len(s.split())`, and string slicing.
</details>

In [ ]:
# ✏️ [EX1]


### Exercise 2: Import and Test (Easy)

Import `string_utils` (from Exercise 1) and write at least 6 assert tests covering normal and edge cases for all three functions.

<details>
<summary>💡 Hint</summary>
Test empty strings, single words, already lowercase strings, strings exactly at max_len.
</details>

In [ ]:
# ✏️ [EX2]


### Exercise 3: sys.path Manipulation (Easy–Medium)

Create a directory called `my_libs/` and write a module `converters.py` inside it with functions:
- `celsius_to_fahrenheit(c)` → F
- `km_to_miles(km)` → miles
- `kg_to_pounds(kg)` → pounds

Add `my_libs/` to `sys.path`, import and test the module.

<details>
<summary>💡 Hint</summary>
Use `os.makedirs("my_libs", exist_ok=True)` and `sys.path.insert(0, ...)`.
</details>

In [ ]:
# ✏️ [EX3]


### Exercise 4: Project Skeleton (Medium)

Create a complete project directory structure for a "weather_analysis" project:
- `src/` with empty `__init__.py`, `loader.py`, `processor.py`, `plotter.py`
- `data/raw/`, `data/processed/`
- `outputs/figures/`, `outputs/reports/`
- `main.py`

Print the directory tree after creation. Each `.py` file should have a docstring and at least one function stub.

<details>
<summary>💡 Hint</summary>
Use a loop with `os.makedirs()` for directories and `%%writefile` for each Python file.
</details>

In [ ]:
# ✏️ [EX4]


### Exercise 5: Multi-Module Pipeline (Medium)

Using the earthquake_analysis project structure, create a pipeline that:
1. Loads 5000 sample records
2. Filters to magnitude 3.0–6.0 and depth 5–80 km
3. Groups by city and computes statistics per city
4. Prints a formatted table with city, count, mean magnitude, and mean depth

All functions must come from imported modules (loader, cleaner, analyzer, reporter).

<details>
<summary>💡 Hint</summary>
Use `analyzer.group_by()` to split by city, then `analyzer.extract_field()` and `analyzer.basic_stats()` per group.
</details>

In [ ]:
# ✏️ [EX5]


### Exercise 6: The `__name__` Guard (Easy–Medium)

Create a module `geometry.py` with functions `circle_area(r)`, `rectangle_area(w, h)`, and `triangle_area(b, h)`. Add a `__name__ == '__main__'` block with at least 6 self-tests.

Show that:
1. Running `!python geometry.py` prints test results
2. Importing `geometry` does NOT print test results

<details>
<summary>💡 Hint</summary>
Put `assert` statements inside the `if __name__ == '__main__':` block.
</details>

In [ ]:
# ✏️ [EX6]


### Exercise 7: Download from URL (Easy–Medium)

Write a function `download_and_preview(url, save_path, n_lines=5)` that:
1. Downloads a file from a URL to save_path
2. Reads the first `n_lines` lines
3. Prints them

Test with a small publicly available CSV or text file. Handle errors gracefully.

<details>
<summary>💡 Hint</summary>
Use `urllib.request.urlretrieve()` for download and `open().readlines()[:n]` for preview.
</details>

In [ ]:
# ✏️ [EX7]


### Exercise 8: Organized Pipeline with Imports (Medium–Challenge)

Create a complete mini-project called `sensor_project` with:
1. `src/data_gen.py` — generates random sensor data (temperature, humidity)
2. `src/cleaner.py` — filters out-of-range values
3. `src/stats.py` — computes statistics per sensor
4. `src/report.py` — formats output as a table

Write a `main.py` that imports all modules and runs the full pipeline. Execute it with `!python sensor_project/main.py`.

<details>
<summary>💡 Hint</summary>
Use `sys.path.insert(0, ...)` in main.py to find the src directory.
</details>

In [ ]:
# ✏️ [EX8]


---
### ⏱️ Checkpoint 5 of 5 — Core Practice (target 04:45)

Run this after Exercises 1–8. It counts only exercise cells that you edited and
ran in this Colab session. It does not inspect correctness or transmit code.


In [ ]:
exercise_checkpoint(5, expected=8)
show_progress_summary()


---
## 🌟 Optional Extension

Exercises 9 and above are optional enrichment. Stop here if the five-hour class has ended.


### Exercise 9: Module Reload Experiment (Easy)

Demonstrate the module caching problem:
1. Create `counter.py` with `COUNT = 1`
2. Import it and print `COUNT`
3. Overwrite `counter.py` with `COUNT = 99`
4. Import again — show that `COUNT` is still 1
5. Use `importlib.reload()` — show that `COUNT` is now 99

<details>
<summary>💡 Hint</summary>
Python caches modules in `sys.modules`. `import` checks the cache first.
</details>

In [ ]:
# ✏️ [EX9]


### Exercise 10: Package with `__init__.py` (Medium)

Create a package (directory with `__init__.py`) called `mytools/` containing:
- `__init__.py` that imports key functions from submodules
- `text.py` with `upper(s)`, `reverse(s)`
- `math_ops.py` with `square(n)`, `cube(n)`

After setup, this should work:
```python
from mytools import upper, reverse, square, cube
```

<details>
<summary>💡 Hint</summary>
In `__init__.py`, write `from .text import upper, reverse` etc. Then add the parent dir to sys.path.
</details>

In [ ]:
# ✏️ [EX10]


### Exercise 11: Config Module Pattern (Medium)

Create a `config.py` module that stores project settings:
- `DATA_DIR`, `OUTPUT_DIR`, `REPORT_DIR` paths
- `DEFAULT_SEED`, `MAX_RECORDS` constants
- A `get_config()` function that returns all settings as a dict

Import config in a pipeline and use its values instead of hard-coded strings.

<details>
<summary>💡 Hint</summary>
Store paths as module-level variables. The `get_config()` function can use `vars()` or a manual dict.
</details>

In [ ]:
# ✏️ [EX11]


### Exercise 12: Testing Imported Functions (Medium)

Create a `test_analyzer.py` module that:
1. Imports functions from `analyzer.py`
2. Runs at least 10 assert-based tests
3. Has a `__name__` guard so tests run when executed directly
4. Prints a summary: "X/Y tests passed"

Run it with `!python test_analyzer.py`.

<details>
<summary>💡 Hint</summary>
Wrap each assert in a try/except to count passes and failures.
</details>

In [ ]:
# ✏️ [EX12]


### Exercise 13: Dynamic Module Loading (Challenge)

Write a function `load_all_modules(directory)` that:
1. Lists all `.py` files in a directory
2. Imports each one dynamically using `importlib.import_module()`
3. Returns a dict mapping module names to module objects
4. Prints a summary of what was loaded

Test by loading all modules from `earthquake_analysis/src/`.

<details>
<summary>💡 Hint</summary>
Use `os.listdir()` to find .py files, strip the extension, and use `importlib.import_module()`.
</details>

In [ ]:
# ✏️ [EX13]


### Exercise 14: Module Documentation Generator (Challenge)

Write a script that:
1. Imports a module
2. Inspects all public functions using `dir()` and `getattr()`
3. Extracts each function's docstring
4. Generates a formatted documentation page with function signatures and descriptions
5. Saves it to a text file

Test on `analyzer.py`.

<details>
<summary>💡 Hint</summary>
Use `inspect.signature(func)` from the `inspect` module to get function parameters.
</details>

In [ ]:
# ✏️ [EX14]


### Exercise 15: Complete Project Scaffold (Challenge)

Create a full project scaffold for a "student_grades" analysis project:
1. Project structure with `src/`, `data/`, `outputs/`, `tests/`
2. Four source modules: `loader.py`, `validator.py`, `analyzer.py`, `reporter.py`
3. Each module has at least 3 functions with docstrings and `__name__` guards
4. A `main.py` that runs the complete pipeline
5. A `tests/test_all.py` that imports and tests functions from all modules
6. Sample data generation

Run `main.py` and `test_all.py` and show both work correctly.

<details>
<summary>💡 Hint</summary>
Plan the data flow first: generate → validate → analyze → report. Each stage should be a separate module.
</details>

In [ ]:
# ✏️ [EX15]


---
### 🌉 Bridge to Next Week

Next week we will learn about **error handling with try/except — making your pipelines robust and graceful when things go wrong** — building on what you learned this week.

Keep practicing and see you in Week 12!